# Tanseek v2 — Student-aware data & conflict-engine starter

This notebook uses the **v2 CSVs** (27 tables: students, registrations, enrollments, holidays, closures) that follow the PostgreSQL v2 schema. It builds relational views and a deterministic constraint checker as a starting point for Menna's scheduling model. Run top to bottom. Keep the `Tanseek_CSV_Data` folder and `constraint_test_cases.csv` beside this notebook. No database or internet is required.

Policy: Saturday–Wednesday, 09:00–17:00, four contiguous 120-minute slots. Availability is due August 31 each year. Data is synthetic; dates in this fixture are for the Fall 2027 term.

In [55]:
from pathlib import Path
import pandas as pd
from app.engine import tables

ROOT = Path.cwd()
DATA = ROOT / "Tanseek_CSV_Data"
CASES = ROOT / "constraint_test_cases.csv"
assert DATA.is_dir() and CASES.is_file(), "Keep Tanseek_CSV_Data/ and constraint_test_cases.csv beside this notebook first."
cases = pd.read_csv(CASES, encoding="utf-8-sig")
pd.DataFrame([(name, len(frame), ", ".join(frame.columns)) for name, frame in sorted(tables.items())], columns=["table", "rows", "columns"])

,table,rows,columns
0,academic_terms,1,"id, name, starts_on, ends_on, state, availabil..."
1,account_department_grants,3,"account_id, department_id, granted_by, created_at"
2,accounts,47,"id, email, password_hash, full_name, role, sta..."
3,allocations,12,"id, term_id, version_id, section_id, instructo..."
4,audit_events,3,"id, actor_account_id, action, entity_type, ent..."
5,availability_slots,33,"submission_id, term_id, slot_id, kind"
6,availability_submissions,8,"id, term_id, instructor_id, state, confirmed_a..."
7,constraint_test_cases,12,"test_case_id, case_name, section_id, instructo..."
8,course_session_requirements,8,"id, course_id, term_id, kind, sessions_per_wee..."
9,courses,4,"id, department_id, code, title, created_by, cr..."


## Relationship checks

These verify the foreign keys needed by the model. A missing reference raises an error before a schedule is evaluated.

In [56]:
relations = [
    ("courses", "department_id", "departments", "id"),
    ("course_session_requirements", "course_id", "courses", "id"),
    ("sections", "requirement_id", "course_session_requirements", "id"),
    ("section_instructors", "section_id", "sections", "id"),
    ("section_instructors", "instructor_id", "accounts", "id"),
    ("students", "department_id", "departments", "id"),
    ("student_group_members", "group_id", "student_groups", "id"),
    ("student_group_members", "student_id", "students", "id"),
    ("student_course_registrations", "student_id", "students", "id"),
    ("student_section_enrollments", "registration_id", "student_course_registrations", "id"),
    ("student_section_enrollments", "section_id", "sections", "id"),
    ("required_equipment", "requirement_id", "course_session_requirements", "id"),
    ("room_equipment", "room_id", "rooms", "id"),
    ("availability_slots", "submission_id", "availability_submissions", "id"),
    ("availability_slots", "slot_id", "time_slots", "id"),
    ("allocations", "section_id", "sections", "id"),
    ("allocations", "start_slot_id", "time_slots", "id"),
]
integrity = []
for child, column, parent, key in relations:
    missing = set(tables[child][column].dropna()) - set(tables[parent][key])
    integrity.append((child + "." + column, parent + "." + key, len(missing)))
integrity = pd.DataFrame(integrity, columns=["foreign_key", "references", "missing_ids"])
assert (integrity.missing_ids == 0).all(), integrity[integrity.missing_ids > 0]
integrity

,foreign_key,references,missing_ids
0,courses.department_id,departments.id,0
1,course_session_requirements.course_id,courses.id,0
2,sections.requirement_id,course_session_requirements.id,0
3,section_instructors.section_id,sections.id,0
4,section_instructors.instructor_id,accounts.id,0
5,students.department_id,departments.id,0
6,student_group_members.group_id,student_groups.id,0
7,student_group_members.student_id,students.id,0
8,student_course_registrations.student_id,students.id,0
9,student_section_enrollments.registration_id,student_course_registrations.id,0


## Connected views

Sections are already bound to one requirement (`sections.requirement_id`). `student_section_enrollments` (ACTIVE) is authoritative for the personal timetable: one LECTURE and one PRACTICAL section per registration. Counts below drive capacity and student-conflict checks. `section_view` gives one row per section.

In [57]:
# v2: sections carry their own requirement; enrollments give authoritative headcounts.
section_requirement_map = {
    int(r.id): int(r.requirement_id) for r in tables["sections"].itertuples()
}
reg_by_student = {
    int(r.id): int(r.student_id) for r in tables["student_course_registrations"].itertuples()
}
active_enr = tables["student_section_enrollments"]
active_enr = active_enr[active_enr["state"] == "ACTIVE"].copy()
active_enr["student_id"] = active_enr["registration_id"].map(reg_by_student)
enrollment_summary = active_enr.groupby("section_id", as_index=False).agg(
    student_ids=("student_id", list), student_count=("student_id", "nunique")
)
group_names = tables["section_group_assignments"].merge(
    tables["student_groups"][["id", "name"]], left_on="group_id", right_on="id", validate="many_to_one"
).groupby("section_id", as_index=False).agg(group_names=("name", list))
section_view = (tables["sections"].merge(tables["courses"][["id", "code", "title", "department_id"]],
             left_on="course_id", right_on="id", validate="many_to_one", suffixes=("", "_course"))
             .merge(enrollment_summary, left_on="id", right_on="section_id", validate="one_to_one", suffixes=("", "_enr"))
             .merge(group_names, left_on="id", right_on="section_id", validate="one_to_one", suffixes=("", "_grp")))
assert len(section_view) == len(tables["sections"])
section_view[["id", "code", "kind", "group_names", "student_count", "max_capacity", "department_id"]].head()

,id,code,kind,group_names,student_count,max_capacity,department_id
0,1,AI301-L1,LECTURE,"[AI Level 3 Group A, AI Level 3 Group B]",16,20,1
1,2,AI301-P1,PRACTICAL,[AI Level 3 Group A],8,8,1
2,3,AI301-P2,PRACTICAL,[AI Level 3 Group B],8,8,1
3,4,AI302-L1,LECTURE,"[AI Level 3 Group A, AI Level 3 Group B]",16,20,1
4,5,AI302-P1,PRACTICAL,[AI Level 3 Group A],8,8,1


In [58]:
# All maps and helpers live in app.engine (single source of truth).
from app.engine import (
    tables, sections, requirements, rooms, slots, accounts,
    section_requirement, groups_by_section,
    students_by_section_ids, students_by_section,
    required, available, eligible,
    submissions, submission_by_staff, availability,
    allocations, minutes, overlaps,
)
section_requirement_map = dict(section_requirement)
print("Engine maps imported from app.engine")

Engine maps imported from app.engine


## Hard constraints

`check_allocation()` returns every reason in a stable order. The primary reason follows the v2 test cases' intended priority (v2 codes: `INVALID_SLOT`, `HOLIDAY`, `INSTRUCTOR_UNAVAILABLE`, `INSTRUCTOR_CONFLICT`, `STUDENT_CONFLICT`). Capacity and conflicts read the authoritative enrollments. An unlisted availability slot is treated as unknown and blocked.

In [59]:
from app.engine import PRIORITY, check_allocation
print("Allocation constraint checker imported from app.engine")

Allocation constraint checker imported from app.engine


## Work through the supplied cases

Each row of `constraint_test_cases.csv` is a `(section, instructor, room, slot, event_date)` proposal. The section's own existing rows are ignored (re-scheduling semantics: moving a section must not self-conflict). `expected_violation` should appear in `reasons`. Case 7 uses LAB-N2: the correct NETWORK_LAB kind with missing equipment, so it reports `EQUIPMENT_SHORTAGE` only.

In [60]:
def resolve_slot(slot_id):
    slot = slots[int(slot_id)]
    return int(slot["term_id"]), int(slot["weekday"]), slot["starts_at"], slot["ends_at"]

results = []
for c in cases.itertuples(index=False):
    section_id = int(c.section_id)
    try:
        term_id, weekday, starts_at, ends_at = resolve_slot(c.slot_id)
    except (KeyError, ValueError, TypeError):
        results.append({"test_case_id": c.test_case_id, "case": c.case_name, "expected": str(c.expected_violation),
                        "actual": "INVALID_SLOT", "all_reasons": ["INVALID_SLOT"],
                        "pass": str(c.expected_violation) == "INVALID_SLOT"})
        continue
    requirement_id = section_requirement_map[section_id]
    base = allocations[allocations["section_id"].astype(int) != section_id].copy()
    result = check_allocation(term_id, section_id, requirement_id, int(c.instructor_id),
                              int(c.room_id), weekday, starts_at, ends_at,
                              existing=base, session_date=str(c.event_date))
    exp = str(c.expected_violation)
    ok = (exp == "NONE" and result["primary_result"] == "FEASIBLE") or (exp in result["reasons"])
    results.append({"test_case_id": c.test_case_id, "case": c.case_name, "expected": exp,
                    "actual": result["primary_result"], "all_reasons": result["reasons"], "pass": ok})
case_results = pd.DataFrame(results)
display(case_results)
assert case_results["pass"].all(), case_results.loc[~case_results["pass"]]

,test_case_id,case,expected,actual,all_reasons,pass
0,1,Valid free allocation,NONE,FEASIBLE,[],True
1,2,Room collision,ROOM_CONFLICT,ROOM_CONFLICT,"[ROOM_CONFLICT, STUDENT_CONFLICT]",True
2,3,Instructor collision,INSTRUCTOR_CONFLICT,INSTRUCTOR_CONFLICT,"[INSTRUCTOR_CONFLICT, STUDENT_CONFLICT, INELIG...",True
3,4,Student collision,STUDENT_CONFLICT,STUDENT_CONFLICT,[STUDENT_CONFLICT],True
4,5,Wrong room type,ROOM_TYPE_MISMATCH,ROOM_TYPE_MISMATCH,"[ROOM_TYPE_MISMATCH, EQUIPMENT_SHORTAGE]",True
5,6,Capacity shortage,CAPACITY_SHORTAGE,CAPACITY_SHORTAGE,[CAPACITY_SHORTAGE],True
6,7,Equipment shortage,EQUIPMENT_SHORTAGE,EQUIPMENT_SHORTAGE,[EQUIPMENT_SHORTAGE],True
7,8,Instructor unavailable,INSTRUCTOR_UNAVAILABLE,INSTRUCTOR_UNAVAILABLE,"[INSTRUCTOR_UNAVAILABLE, ROOM_CONFLICT]",True
8,9,Room closed,ROOM_CLOSED,ROOM_CLOSED,[ROOM_CLOSED],True
9,10,Holiday,HOLIDAY,HOLIDAY,[HOLIDAY],True


## Generate feasible candidates and rank alternatives

The demo generates candidate combinations for one section and one weekly session. It excludes hard conflicts and prefers a `PREFERRED` availability slot, then an earlier time. For a full timetable solver, add weekly session instances, teacher workload, room closure/holiday tables, and an optimizer that chooses a compatible set *jointly*. This starter does not claim to solve the global optimization problem.

In [61]:
def rank_candidates(section_id, requirement_id, limit=10):
    sec = sections[int(section_id)]
    term_id = int(sec["term_id"])
    staff = sorted({i for s, r, i in eligible if s == int(section_id) and r == int(requirement_id)})
    rows = []
    for instructor_id in staff:
        submission = submission_by_staff.get((term_id, instructor_id))
        for slot_id, slot in slots.items():
            if int(slot["term_id"]) != term_id: continue
            for room_id in rooms:
                result = check_allocation(term_id, section_id, requirement_id, instructor_id,
                    room_id, slot["weekday"], slot["starts_at"], slot["ends_at"])
                if result["primary_result"] != "FEASIBLE": continue
                kind = availability.get((int(submission.id), slot_id), "UNKNOWN") if submission else "UNKNOWN"
                rows.append({"section_id": section_id, "requirement_id": requirement_id,
                             "instructor_id": instructor_id, "room_id": room_id,
                             "slot_id": slot_id, "weekday": slot["weekday"],
                             "starts_at": slot["starts_at"], "availability": kind,
                             "preference_rank": 0 if kind == "PREFERRED" else 1})
    if not rows: return pd.DataFrame(columns=["section_id", "requirement_id", "instructor_id", "room_id", "slot_id", "weekday", "starts_at", "availability", "preference_rank"])
    return pd.DataFrame(rows).sort_values(["preference_rank", "weekday", "starts_at", "room_id"]).head(limit).reset_index(drop=True)

candidate_view = rank_candidates(section_id=2, requirement_id=2)
candidate_view

,section_id,requirement_id,instructor_id,room_id,slot_id,weekday,starts_at,availability,preference_rank
0,2,2,12,3,2,6,11:00,AVAILABLE,1
1,2,2,12,4,2,6,11:00,AVAILABLE,1
2,2,2,12,3,3,6,13:00,AVAILABLE,1
3,2,2,12,4,3,6,13:00,AVAILABLE,1
4,2,2,12,3,6,7,11:00,AVAILABLE,1
5,2,2,12,4,6,7,11:00,AVAILABLE,1
6,2,2,12,3,7,7,13:00,AVAILABLE,1
7,2,2,12,4,7,7,13:00,AVAILABLE,1


In [62]:
from ortools.sat.python import cp_model
from collections import defaultdict, Counter
import json

print("OR-Tools ready ")

OR-Tools ready 


In [63]:
def run_fixture_tests():
    results = []

    for c in cases.itertuples(index=False):
        section_id = int(c.section_id)
        try:
            term_id, weekday, starts_at, ends_at = resolve_slot(c.slot_id)
        except (KeyError, ValueError, TypeError):
            exp = str(c.expected_violation)
            results.append({
                "test_case_id": c.test_case_id,
                "expected": exp,
                "actual": "INVALID_SLOT",
                "all_reasons": ["INVALID_SLOT"],
                "pass": exp == "INVALID_SLOT"
            })
            continue
        requirement_id = section_requirement_map[section_id]
        base = allocations[allocations["section_id"].astype(int) != section_id].copy()

        result = check_allocation(
            term_id,
            section_id,
            requirement_id,
            int(c.instructor_id),
            int(c.room_id),
            weekday,
            starts_at,
            ends_at,
            existing=base,
            session_date=str(c.event_date)
        )

        exp = str(c.expected_violation)
        results.append({
            "test_case_id": c.test_case_id,
            "expected": exp,
            "actual": result["primary_result"],
            "all_reasons": result["reasons"],
            "pass": (exp == "NONE" and result["primary_result"] == "FEASIBLE") or (exp in result["reasons"])
        })

    result_df = pd.DataFrame(results)

    display(result_df)

    assert result_df["pass"].all(), \
        result_df.loc[~result_df["pass"]]

    print("All 12 fixture tests passed")

    return result_df

In [64]:
# -------------------------
# Schedule Version Context
# -------------------------

TERM_ID = int(
    tables["sections"]["term_id"]
    .mode()
    .iloc[0]
)

schedule_versions_df = (
    tables["schedule_versions"]
    .copy()
)

term_versions = (
    schedule_versions_df[
        schedule_versions_df["term_id"].astype(int)
        == TERM_ID
    ]
    .sort_values(
        ["version_number", "id"]
    )
    .reset_index(drop=True)
)

display(
    term_versions[
        [
            "id",
            "term_id",
            "version_number",
            "name",
            "state"
        ]
    ]
)

,id,term_id,version_number,name,state
0,1,1,1,Baseline v1,PUBLISHED


In [65]:
from app.engine import (
    get_latest_version_by_state, published_version, working_version,
)


print(
    "Published version:",
    None
    if published_version is None
    else published_version["name"]
)

print(
    "Working version:",
    None
    if working_version is None
    else working_version["name"]
)

Published version: Baseline v1
Working version: None


In [66]:
from app.engine import (
    get_version_allocations, published_allocations, working_allocations,
    published_version,
)


print(
    "Published allocations:",
    len(published_allocations)
)

print(
    "Working allocations:",
    len(working_allocations)
)

Published allocations: 12
Working allocations: 0


In [67]:
# FRESH solver mode for v2 lives in app.engine: the published baseline
# stays immutable for change validation, but the solver schedules
# from the DRAFT working set only (empty here: everything from scratch).
from app.engine import solver_existing_allocations


print(
    "Solver existing allocations:",
    len(solver_existing_allocations)
)

Solver existing allocations: 0


In [68]:
from app.engine import (
    published_snapshot, published_allocations, verify_published_immutable,
)


verify_published_immutable()

True

In [69]:
from app.engine import term_holidays, room_closures


print(
    "Term holidays loaded:",
    len(term_holidays)
)

print(
    "Room closures loaded:",
    len(room_closures)
)

Term holidays loaded: 1
Room closures loaded: 1


In [70]:
from app.engine import is_term_holiday, is_room_closed
print("Holiday / closure helpers imported from app.engine")

Holiday / closure helpers imported from app.engine


In [71]:
from app.engine import check_candidate_strict
print("Strict candidate checker imported from app.engine")

Strict candidate checker imported from app.engine


In [72]:
from app.engine import evaluate_published_change_request
print("Published change-request validator imported from app.engine")

Published change-request validator imported from app.engine


In [73]:
# -------------------------
# Date Constraint Tests (real v2 rows)
# -------------------------

TEST_DATE = "2027-10-03"

# v2 fixture: 2027-10-06 is a term holiday; room 3 is closed 2027-11-15 11:00-13:00Z.
assert is_term_holiday(
    TERM_ID,
    "2027-10-06"
)

assert not is_term_holiday(
    TERM_ID,
    "2027-10-03"
)

assert is_room_closed(
    room_id=3,
    session_date="2027-11-15",
    starts_at="11:00",
    ends_at="13:00"
)

assert not is_room_closed(
    room_id=3,
    session_date="2027-11-15",
    starts_at="13:00",
    ends_at="15:00"
)


print(
    "Task 4 holiday/closure tests passed "
)

Task 4 holiday/closure tests passed 


In [74]:
# -------------------------
# Published Change Request / Immutability Test
# -------------------------

def test_published_change_request():

    # Current synthetic data may not contain
    # a real PUBLISHED schedule yet.
    if published_version is None:

        result = evaluate_published_change_request(
            allocation_id=-1,
            session_date=TEST_DATE
        )

        assert (
            result["status"]
            == "NO_PUBLISHED_VERSION"
        )

        assert (
            result["reasons"]
            == ["NO_PUBLISHED_VERSION"]
        )

        print(
            "Published change-request gate test passed ✅"
        )

        return


    assert not published_snapshot.empty

    before = (
        published_snapshot
        .copy(deep=True)
        .reset_index(drop=True)
    )

    target = published_snapshot.iloc[0]

    # Pick a concrete date with the same weekday
    # as the published allocation.
    probe_date = pd.Timestamp(TEST_DATE)

    required_weekday = int(
        target["weekday"]
    )

    delta_days = (
        required_weekday
        - probe_date.isoweekday()
    ) % 7

    probe_date = (
        probe_date
        + pd.Timedelta(days=delta_days)
    ).date()


    result = evaluate_published_change_request(
        allocation_id=int(target["id"]),
        session_date=str(probe_date)
    )

    # A change request may be feasible or blocked,
    # but it must never mutate the published source.
    assert result["status"] in (
        "FEASIBLE",
        "BLOCKED"
    )

    after = (
        published_snapshot
        .copy(deep=True)
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        before,
        after
    )

    verify_published_immutable()

    print(
        "Published change-request immutability test passed ✅"
    )


test_published_change_request()

Published change-request immutability test passed ✅


In [75]:
# v2 sections are already bound to one requirement each.
section_requirements_df = (
    tables["sections"][["id", "term_id", "course_id", "code", "requirement_id", "kind"]]
    .rename(columns={"id": "section_id", "code": "section_code"})
    .merge(
        tables["course_session_requirements"][["id", "sessions_per_week"]].rename(
            columns={"id": "requirement_id"}
        ),
        on=["requirement_id"],
        validate="many_to_one",
    )
)


print(
    "Section requirements:",
    len(section_requirements_df)
)

display(
    section_requirements_df.head(12)
)

Section requirements: 12


,section_id,term_id,course_id,section_code,requirement_id,kind,sessions_per_week
0,1,1,1,AI301-L1,1,LECTURE,1
1,2,1,1,AI301-P1,2,PRACTICAL,1
2,3,1,1,AI301-P2,2,PRACTICAL,1
3,4,1,2,AI302-L1,3,LECTURE,1
4,5,1,2,AI302-P1,4,PRACTICAL,1
5,6,1,2,AI302-P2,4,PRACTICAL,1
6,7,1,3,CS301-L1,5,LECTURE,1
7,8,1,3,CS301-P1,6,PRACTICAL,1
8,9,1,3,CS301-P2,6,PRACTICAL,1
9,10,1,4,CS302-L1,7,LECTURE,1


In [76]:
from app.solver import build_weekly_sessions


weekly_sessions = build_weekly_sessions()

print("Sessions still needing allocation:", len(weekly_sessions))

display(weekly_sessions.head(20))

Sessions still needing allocation: 12


,session_key,term_id,section_id,section_code,requirement_id,course_id,kind,instance_number,duration_minutes,required_room_kind
0,S1_R1_W1,1,1,AI301-L1,1,1,LECTURE,1,120,LECTURE_HALL
1,S2_R2_W1,1,2,AI301-P1,2,1,PRACTICAL,1,120,COMPUTER_LAB
2,S3_R2_W1,1,3,AI301-P2,2,1,PRACTICAL,1,120,COMPUTER_LAB
3,S4_R3_W1,1,4,AI302-L1,3,2,LECTURE,1,120,LECTURE_HALL
4,S5_R4_W1,1,5,AI302-P1,4,2,PRACTICAL,1,120,GPU_LAB
5,S6_R4_W1,1,6,AI302-P2,4,2,PRACTICAL,1,120,GPU_LAB
6,S7_R5_W1,1,7,CS301-L1,5,3,LECTURE,1,120,LECTURE_HALL
7,S8_R6_W1,1,8,CS301-P1,6,3,PRACTICAL,1,120,COMPUTER_LAB
8,S9_R6_W1,1,9,CS301-P2,6,3,PRACTICAL,1,120,COMPUTER_LAB
9,S10_R7_W1,1,10,CS302-L1,7,4,LECTURE,1,120,LECTURE_HALL


In [77]:
from app.solver import SOFT_SCORE_DEFINITIONS

SOFT_SCORE_DEFINITIONS

{'STAFF_PREFERENCE': {'max_points': 30, 'scope': 'CANDIDATE'},
 'CAPACITY_FIT': {'max_points': 25, 'scope': 'CANDIDATE'},
 'EQUIPMENT_MATCH': {'max_points': 20, 'scope': 'CANDIDATE'},
 'COMPACTNESS': {'points_per_adjacent_pair': 15, 'scope': 'SCHEDULE'},
 'ROOM_UTILIZATION': {'penalty_per_active_room': 10, 'scope': 'SCHEDULE'}}

In [78]:
print("Required weekly sessions:", section_requirements_df["sessions_per_week"].sum())
print("Already allocated:", len(solver_existing_allocations))
print("Still to schedule:", len(weekly_sessions))

Required weekly sessions: 12
Already allocated: 0
Still to schedule: 12


In [79]:
from app.solver import generate_all_candidates
print("Candidate generator imported from app.solver")

Candidate generator imported from app.solver


In [80]:
candidates, candidate_diagnostics = (
    generate_all_candidates(
        weekly_sessions
    )
)

print(
    "Total feasible candidates:",
    len(candidates)
)

display(
    candidates.head(20)
)

Total feasible candidates: 82


,candidate_id,session_key,term_id,section_id,section_code,requirement_id,instance_number,instructor_id,room_id,room_capacity,...,ends_at,availability,student_count,staff_preference_score,capacity_fit_pct,capacity_fit_score,equipment_fit_pct,equipment_match_score,total_score,soft_reason_codes
0,1,S1_R1_W1,1,1,AI301-L1,1,1,8,1,40,...,11:00,AVAILABLE,16,0,40,10,100,20,30,"[CAPACITY_FIT, EQUIPMENT_MATCH]"
1,2,S1_R1_W1,1,1,AI301-L1,1,1,8,2,35,...,11:00,AVAILABLE,16,0,46,12,100,20,32,"[CAPACITY_FIT, EQUIPMENT_MATCH]"
2,3,S1_R1_W1,1,1,AI301-L1,1,1,8,1,40,...,13:00,AVAILABLE,16,0,40,10,100,20,30,"[CAPACITY_FIT, EQUIPMENT_MATCH]"
3,4,S1_R1_W1,1,1,AI301-L1,1,1,8,2,35,...,13:00,AVAILABLE,16,0,46,12,100,20,32,"[CAPACITY_FIT, EQUIPMENT_MATCH]"
4,5,S1_R1_W1,1,1,AI301-L1,1,1,8,1,40,...,11:00,AVAILABLE,16,0,40,10,100,20,30,"[CAPACITY_FIT, EQUIPMENT_MATCH]"
5,6,S1_R1_W1,1,1,AI301-L1,1,1,8,2,35,...,11:00,AVAILABLE,16,0,46,12,100,20,32,"[CAPACITY_FIT, EQUIPMENT_MATCH]"
6,7,S1_R1_W1,1,1,AI301-L1,1,1,8,1,40,...,13:00,AVAILABLE,16,0,40,10,100,20,30,"[CAPACITY_FIT, EQUIPMENT_MATCH]"
7,8,S1_R1_W1,1,1,AI301-L1,1,1,8,2,35,...,13:00,AVAILABLE,16,0,46,12,100,20,32,"[CAPACITY_FIT, EQUIPMENT_MATCH]"
8,9,S2_R2_W1,1,2,AI301-P1,2,1,12,3,20,...,13:00,AVAILABLE,8,0,40,10,40,8,18,"[CAPACITY_FIT, EQUIPMENT_MATCH]"
9,10,S2_R2_W1,1,2,AI301-P1,2,1,12,4,20,...,13:00,AVAILABLE,8,0,40,10,40,8,18,"[CAPACITY_FIT, EQUIPMENT_MATCH]"


In [81]:
candidate_counts = (
    candidates
    .groupby("session_key")
    .size()
    .rename("candidate_count")
    .reset_index()
)

coverage = (
    weekly_sessions[
        [
            "session_key",
            "section_code",
            "kind"
        ]
    ]
    .merge(
        candidate_counts,
        on="session_key",
        how="left"
    )
)

coverage["candidate_count"] = (
    coverage["candidate_count"]
    .fillna(0)
    .astype(int)
)

display(coverage)

,session_key,section_code,kind,candidate_count
0,S1_R1_W1,AI301-L1,LECTURE,8
1,S2_R2_W1,AI301-P1,PRACTICAL,8
2,S3_R2_W1,AI301-P2,PRACTICAL,8
3,S4_R3_W1,AI302-L1,LECTURE,8
4,S5_R4_W1,AI302-P1,PRACTICAL,4
5,S6_R4_W1,AI302-P2,PRACTICAL,4
6,S7_R5_W1,CS301-L1,LECTURE,10
7,S8_R6_W1,CS301-P1,PRACTICAL,8
8,S9_R6_W1,CS301-P2,PRACTICAL,8
9,S10_R7_W1,CS302-L1,LECTURE,8


In [82]:
problem_sessions = coverage[
    coverage["candidate_count"] == 0
]

print(
    "Sessions with zero candidates:",
    len(problem_sessions)
)

if not problem_sessions.empty:
    display(problem_sessions)

    display(
        candidate_diagnostics[
            candidate_diagnostics[
                "session_key"
            ].isin(
                problem_sessions[
                    "session_key"
                ]
            )
        ]
        .sort_values(
            ["session_key", "count"],
            ascending=[True, False]
        )
    )
else:
    print(
        "Every weekly session has at least one feasible candidate ✅"
    )

Sessions with zero candidates: 0
Every weekly session has at least one feasible candidate ✅


In [83]:
from app.solver import create_decision_variables, UNSCHEDULED_PENALTY


model, x, unscheduled = create_decision_variables(
    candidates, weekly_sessions
)

print("Decision variables created")

Decision variables created


In [84]:
from app.solver import add_session_coverage_constraints


add_session_coverage_constraints(
    model, x, unscheduled, candidates, weekly_sessions
)

print("Session assignment constraints added")

Session assignment constraints added


In [85]:
from app.solver import add_room_conflict_constraints


add_room_conflict_constraints(model, x, candidates)

print("Room conflict constraints added")

Room conflict constraints added


In [86]:
from app.solver import add_instructor_conflict_constraints


add_instructor_conflict_constraints(model, x, candidates)

print("Instructor conflict constraints added")

Instructor conflict constraints added


In [87]:
from app.solver import add_section_conflict_constraints


add_section_conflict_constraints(model, x, candidates)

print("Section conflict constraints added")

Section conflict constraints added


In [88]:
from app.solver import add_student_conflict_constraints


add_student_conflict_constraints(model, x, candidates)

print("Individual-student conflict constraints added")

Individual-student conflict constraints added


In [89]:
from app.solver import (
    build_candidate_score, find_adjacent_slots,
    build_occupancy_variables, build_compactness_terms,
    build_room_utilization_terms, build_unscheduled_cost,
    set_final_objective, COMPACTNESS_POINTS,
    ROOM_ACTIVE_PENALTY, UNSCHEDULED_PENALTY,
)


candidate_score = build_candidate_score(candidates, x)

adjacent_slots = find_adjacent_slots()

section_occupancy, instructor_occupancy = build_occupancy_variables(
    model, x, candidates
)

compactness_vars, compactness_bonus = build_compactness_terms(
    model, candidates, section_occupancy,
    instructor_occupancy, adjacent_slots
)

room_used, room_utilization_penalty = build_room_utilization_terms(
    model, x, candidates
)

unscheduled_cost = build_unscheduled_cost(unscheduled)

set_final_objective(
    model, candidate_score, compactness_bonus,
    room_utilization_penalty, unscheduled_cost
)


print(
    "Global objective created"
)

Global objective created


In [90]:
from app.solver import solve_model


status, status_name, solver = solve_model(model, 30, 4)

print(
    "Solver status:",
    status_name
)

Solver status: OPTIMAL


In [91]:
from app.solver import extract_solution


details = extract_solution(
    solver, x, unscheduled, candidates,
    weekly_sessions, compactness_vars, room_used,
)

selected_schedule = details["selected_schedule"]
unscheduled_sessions = details["unscheduled_sessions"]


print(
    "Scheduled sessions:",
    len(selected_schedule)
)

print(
    "Unscheduled sessions:",
    len(unscheduled_sessions)
)

print(
    "Unscheduled keys:",
    unscheduled_sessions
)

display(selected_schedule)

Scheduled sessions: 12
Unscheduled sessions: 0
Unscheduled keys: []


,candidate_id,session_key,term_id,section_id,section_code,requirement_id,instance_number,instructor_id,room_id,room_capacity,...,ends_at,availability,student_count,staff_preference_score,capacity_fit_pct,capacity_fit_score,equipment_fit_pct,equipment_match_score,total_score,soft_reason_codes
0,51,S8_R6_W1,1,8,CS301-P1,6,1,14,3,20,...,13:00,AVAILABLE,8,0,40,10,70,14,24,"[CAPACITY_FIT, EQUIPMENT_MATCH]"
1,79,S12_R8_W1,1,12,CS302-P2,8,1,15,6,16,...,13:00,AVAILABLE,8,0,50,12,50,10,22,"[CAPACITY_FIT, EQUIPMENT_MATCH]"
2,61,S9_R6_W1,1,9,CS301-P2,6,1,15,3,20,...,15:00,AVAILABLE,8,0,40,10,70,14,24,"[CAPACITY_FIT, EQUIPMENT_MATCH]"
3,76,S11_R8_W1,1,11,CS302-P1,8,1,14,6,16,...,15:00,AVAILABLE,8,0,50,12,50,10,22,"[CAPACITY_FIT, EQUIPMENT_MATCH]"
4,74,S10_R7_W1,1,10,CS302-L1,7,1,11,2,35,...,13:00,AVAILABLE,16,0,46,12,100,20,32,"[CAPACITY_FIT, EQUIPMENT_MATCH]"
5,50,S7_R5_W1,1,7,CS301-L1,5,1,10,2,35,...,11:00,AVAILABLE,16,0,46,12,100,20,32,"[CAPACITY_FIT, EQUIPMENT_MATCH]"
6,9,S2_R2_W1,1,2,AI301-P1,2,1,12,3,20,...,13:00,AVAILABLE,8,0,40,10,40,8,18,"[CAPACITY_FIT, EQUIPMENT_MATCH]"
7,37,S6_R4_W1,1,6,AI302-P2,4,1,13,5,16,...,13:00,AVAILABLE,8,0,50,12,50,10,22,"[CAPACITY_FIT, EQUIPMENT_MATCH]"
8,19,S3_R2_W1,1,3,AI301-P2,2,1,13,3,20,...,15:00,AVAILABLE,8,0,40,10,40,8,18,"[CAPACITY_FIT, EQUIPMENT_MATCH]"
9,34,S5_R4_W1,1,5,AI302-P1,4,1,12,5,16,...,15:00,AVAILABLE,8,0,50,12,50,10,22,"[CAPACITY_FIT, EQUIPMENT_MATCH]"


In [92]:
from app.solver import build_unscheduled_report


unscheduled_rows = build_unscheduled_report(
    weekly_sessions, candidate_diagnostics, unscheduled_sessions
)


unscheduled_report = pd.DataFrame(
    unscheduled_rows
)

display(unscheduled_report)

""


In [93]:
from app.solver import get_ranked_alternatives
print("Ranked alternatives imported from app.solver")

Ranked alternatives imported from app.solver


In [94]:
alternatives = get_ranked_alternatives(
    selected_schedule, candidates, "S2_R2_W1"
)

print(
    "Number of alternatives:",
    len(alternatives)
)

for i, alt in enumerate(
    alternatives,
    start=1
):
    print(
        i,
        "Room:", alt["room_id"],
        "| Instructor:", alt["instructor_id"],
        "| Slot:", alt["slot_id"],
        "| Availability:", alt["availability"]
    )

Number of alternatives: 3
1 Room: 4 | Instructor: 12 | Slot: 2 | Availability: AVAILABLE
2 Room: 3 | Instructor: 12 | Slot: 7 | Availability: AVAILABLE
3 Room: 4 | Instructor: 12 | Slot: 7 | Availability: AVAILABLE


In [95]:
def calculate_room_score(
    room_id,
    section_id,
    requirement_id
):
    room = rooms[int(room_id)]

    student_count = students_by_section.get(
        int(section_id),
        0
    )

    room_capacity = int(room["capacity"])

    # 1) Capacity Fit - 50
    if room_capacity < student_count:
        capacity_score = 0
    else:
        capacity_ratio = student_count / room_capacity
        capacity_score = 50 * capacity_ratio

    # 2) Equipment Match - 30
    required_items = [
        (equipment_id, quantity)
        for (req_id, equipment_id), quantity in required.items()
        if int(req_id) == int(requirement_id)
    ]

    if not required_items:
        equipment_score = 30
    else:
        matched = 0

        for equipment_id, required_quantity in required_items:

            available_quantity = available.get(
                (
                    int(room_id),
                    int(equipment_id)
                ),
                0
            )

            if available_quantity >= required_quantity:
                matched += 1

        equipment_score = (
            30 * matched / len(required_items)
        )

    # 3) Room Type - 20
    requirement = requirements[int(requirement_id)]

    if room["kind"] == requirement["required_room_kind"]:
        room_type_score = 20
    else:
        room_type_score = 0

    # Final Score
    room_score = (
        capacity_score
        + equipment_score
        + room_type_score
    )

    return round(room_score, 2)
print(
    calculate_room_score(
        room_id=3,
        section_id=2,
        requirement_id=2
    )
)

70.0


In [96]:
candidates["room_score"] = candidates.apply(
    lambda row: calculate_room_score(
        room_id=row["room_id"],
        section_id=row["section_id"],
        requirement_id=row["requirement_id"]
    ),
    axis=1
)
display(
    candidates[
        [
            "session_key",
            "room_id",
            "instructor_id",
            "slot_id",
            "room_score"
        ]
    ]
    .sort_values(
        "room_score",
        ascending=False
    )
    .head(20)
)

,session_key,room_id,instructor_id,slot_id,room_score
32,S5_R4_W1,5,12,2,75.00
33,S5_R4_W1,5,12,3,75.00
74,S11_R8_W1,6,14,10,75.00
75,S11_R8_W1,6,14,11,75.00
76,S11_R8_W1,6,14,14,75.00
77,S11_R8_W1,6,14,15,75.00
78,S12_R8_W1,6,15,10,75.00
79,S12_R8_W1,6,15,11,75.00
80,S12_R8_W1,6,15,14,75.00
81,S12_R8_W1,6,15,15,75.00


In [97]:
get_ranked_alternatives(
    selected_schedule, candidates, "S2_R2_W1"
)

[{'candidate_id': 10,
  'instructor_id': 12,
  'room_id': 4,
  'slot_id': 2,
  'weekday': 6,
  'starts_at': '11:00',
  'ends_at': '13:00',
  'availability': 'AVAILABLE',
  'score': 18,
  'score_breakdown': {'staff_preference': 0,
   'capacity_fit': 10,
   'capacity_fit_pct': 40,
   'equipment_match': 8,
   'equipment_fit_pct': 40},
  'soft_reason_codes': ['CAPACITY_FIT', 'EQUIPMENT_MATCH']},
 {'candidate_id': 15,
  'instructor_id': 12,
  'room_id': 3,
  'slot_id': 7,
  'weekday': 7,
  'starts_at': '13:00',
  'ends_at': '15:00',
  'availability': 'AVAILABLE',
  'score': 18,
  'score_breakdown': {'staff_preference': 0,
   'capacity_fit': 10,
   'capacity_fit_pct': 40,
   'equipment_match': 8,
   'equipment_fit_pct': 40},
  'soft_reason_codes': ['CAPACITY_FIT', 'EQUIPMENT_MATCH']},
 {'candidate_id': 16,
  'instructor_id': 12,
  'room_id': 4,
  'slot_id': 7,
  'weekday': 7,
  'starts_at': '13:00',
  'ends_at': '15:00',
  'availability': 'AVAILABLE',
  'score': 18,
  'score_breakdown': {'st

In [98]:
alternatives = get_ranked_alternatives(
    selected_schedule, candidates, "S2_R2_W1"
)

print("Number of alternatives:", len(alternatives))

for i, alt in enumerate(alternatives, start=1):
    print(
        i,
        "Room:", alt["room_id"],
        "| Instructor:", alt["instructor_id"],
        "| Slot:", alt["slot_id"],
        "| Score:", alt["score"]
    )

Number of alternatives: 3
1 Room: 4 | Instructor: 12 | Slot: 2 | Score: 18
2 Room: 3 | Instructor: 12 | Slot: 7 | Score: 18
3 Room: 4 | Instructor: 12 | Slot: 7 | Score: 18


In [99]:
# -------------------------
# Final Objective Metrics
# -------------------------

candidate_score_value = (
    int(selected_schedule["total_score"].sum())
    if not selected_schedule.empty
    else 0
)


selected_compact_pairs = int(
    sum(
        solver.Value(variable)
        for variable in compactness_vars
    )
)


compactness_bonus_value = (
    selected_compact_pairs
    * COMPACTNESS_POINTS
)


active_room_ids = [
    int(room_id)

    for room_id, variable
    in room_used.items()

    if solver.Value(variable) == 1
]


active_room_count = len(
    active_room_ids
)


room_utilization_penalty_value = (
    active_room_count
    * ROOM_ACTIVE_PENALTY
)


unscheduled_penalty_value = (
    len(unscheduled_sessions)
    * UNSCHEDULED_PENALTY
)


final_objective_value = (
    candidate_score_value
    + compactness_bonus_value
    - room_utilization_penalty_value
    - unscheduled_penalty_value
)


print(
    "Candidate score:",
    candidate_score_value
)

print(
    "Compact adjacent pairs:",
    selected_compact_pairs
)

print(
    "Compactness bonus:",
    compactness_bonus_value
)

print(
    "Active rooms:",
    active_room_count
)

print(
    "Room utilization penalty:",
    room_utilization_penalty_value
)

print(
    "Final objective value:",
    final_objective_value
)

Candidate score: 300
Compact adjacent pairs: 4
Compactness bonus: 60
Active rooms: 4
Room utilization penalty: 40
Final objective value: 320


In [100]:
from app.solver import build_scheduled_payload


scheduled_payload = build_scheduled_payload(
    selected_schedule, candidates
)


print(
    "Scheduled payload ready"
)

Scheduled payload ready


In [101]:
from app.engine import find_schedule_conflicts


post_solve_conflicts = find_schedule_conflicts(
    selected_schedule
)


assert not post_solve_conflicts, post_solve_conflicts


print(
    "Post-solve hard conflicts: 0"
)

Post-solve hard conflicts: 0


In [102]:
from app.engine import (
    TERM_ID, working_version, published_version,
    term_holidays, room_closures,
)
from app.solver import SOFT_SCORE_DEFINITIONS
from app.engine import PRIORITY, solver_existing_allocations


solver_payload = {
    "solver_status": status_name,

    "version_context": {
        "term_id": int(TERM_ID),

        "working_version": (
            None
            if working_version is None
            else {
                "id": int(working_version["id"]),
                "name": working_version["name"],
                "version_number": int(
                    working_version["version_number"]
                ),
                "state": working_version["state"]
            }
        ),

        "published_version": (
            None
            if published_version is None
            else {
                "id": int(published_version["id"]),
                "name": published_version["name"],
                "version_number": int(
                    published_version["version_number"]
                ),
                "state": published_version["state"]
            }
        ),

        "published_schedule_immutable_enforced": True,

        "published_version_available": published_version is not None,
    },

    "summary": {
        "required_weekly_sessions": int(
            len(weekly_sessions) + len(solver_existing_allocations)
        ),

        "existing_allocations": int(len(solver_existing_allocations)),

        "solver_sessions": int(len(weekly_sessions)),

        "scheduled": int(len(selected_schedule)),

        "unscheduled": int(len(unscheduled_sessions)),
    },

    "hard_reason_codes": PRIORITY,
    "objective_breakdown": {
        "candidate_level_score": details["candidate_score_value"],

        "compactness": {
            "adjacent_pairs": details["selected_compact_pairs"],

            "points_per_pair": COMPACTNESS_POINTS,

            "bonus": details["compactness_bonus_value"],
        },

        "room_utilization": {
            "active_room_count": details["active_room_count"],

            "active_room_ids": details["active_room_ids"],

            "penalty_per_active_room": ROOM_ACTIVE_PENALTY,

            "penalty": details["room_penalty_value"],
        },

        "unscheduled": {
            "count": int(len(unscheduled_sessions)),

            "penalty_per_session": UNSCHEDULED_PENALTY,

            "total_penalty": details["unscheduled_penalty_value"],
        },

        "final_objective_value": details["calculated_objective"],
    },

    "soft_score_definitions": SOFT_SCORE_DEFINITIONS,

    "date_specific_constraints": {
        "mode": "CONCRETE_DATE_CHANGE_REQUEST",

        "term_holidays_loaded": int(len(term_holidays)),

        "room_closures_loaded": int(len(room_closures)),
    },

    "scheduled_sessions": scheduled_payload,

    "unscheduled_sessions": unscheduled_rows,
}

In [103]:
print(
    json.dumps(
        solver_payload,
        indent=2,
        ensure_ascii=False
    )[:5000]
)

{
  "solver_status": "OPTIMAL",
  "version_context": {
    "term_id": 1,
    "working_version": null,
    "published_version": {
      "id": 1,
      "name": "Baseline v1",
      "version_number": 1,
      "state": "PUBLISHED"
    },
    "published_schedule_immutable_enforced": true,
    "published_version_available": true
  },
  "summary": {
    "required_weekly_sessions": 12,
    "existing_allocations": 0,
    "solver_sessions": 12,
    "scheduled": 12,
    "unscheduled": 0
  },
  "hard_reason_codes": [
    "INVALID_SLOT",
    "HOLIDAY",
    "ROOM_CLOSED",
    "INVALID_DURATION",
    "AVAILABILITY_NOT_CONFIRMED",
    "INSTRUCTOR_UNAVAILABLE",
    "ROOM_CONFLICT",
    "INSTRUCTOR_CONFLICT",
    "STUDENT_CONFLICT",
    "GROUP_CONFLICT",
    "ROOM_TYPE_MISMATCH",
    "CAPACITY_SHORTAGE",
    "EQUIPMENT_SHORTAGE",
    "INELIGIBLE_INSTRUCTOR",
    "INVALID_REFERENCE"
  ],
  "objective_breakdown": {
    "candidate_level_score": 300,
    "compactness": {
      "adjacent_pairs": 4,
      "po

In [104]:
OUTPUT_FILE = (
    ROOT / "tanseek_solver_output.json"
)

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        solver_payload,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Saved:",
    OUTPUT_FILE.resolve()
)

Saved: C:\Users\engme\OneDrive\Desktop\Tanseek_Menna_Model_v2_Starter\tanseek_solver_output.json


In [105]:
# -------------------------
# Final Validation
# -------------------------

fixture_results = run_fixture_tests()

verify_published_immutable()

# All fixture tests must pass
assert fixture_results["pass"].all()

# Every required solver session must be either
# scheduled or explicitly unscheduled
assert (
    len(selected_schedule)
    + len(unscheduled_sessions)
    == len(weekly_sessions)
)

scheduled_keys = set(
    selected_schedule["session_key"]
)

unscheduled_keys = set(
    unscheduled_sessions
)

required_keys = set(
    weekly_sessions["session_key"]
)

# No session can be both scheduled and unscheduled
assert scheduled_keys.isdisjoint(
    unscheduled_keys
)

# No session can disappear
assert (
    scheduled_keys | unscheduled_keys
    == required_keys
)

# JSON file must exist
assert OUTPUT_FILE.exists()
assert len(post_solve_conflicts) == 0

print()
print("===================================")
print("TANSEEK MODEL FINAL CHECK ✅")
print("===================================")

print(
    "Fixture tests:",
    f"{fixture_results['pass'].sum()}/"
    f"{len(fixture_results)}"
)

print(
    "Existing allocations:",
    len(solver_existing_allocations)
)

print(
    "Weekly sessions to solve:",
    len(weekly_sessions)
)

print(
    "Scheduled:",
    len(selected_schedule)
)

print(
    "Unscheduled:",
    len(unscheduled_sessions)
)

print(
    "Term holidays loaded:",
    len(term_holidays)
)

print(
    "Room closures loaded:",
    len(room_closures)
)

print(
    "JSON output:",
    OUTPUT_FILE.resolve()
)
print(
    "Post-solve hard conflicts:",
    len(post_solve_conflicts)
)

print("===================================")

,test_case_id,expected,actual,all_reasons,pass
0,1,NONE,FEASIBLE,[],True
1,2,ROOM_CONFLICT,ROOM_CONFLICT,"[ROOM_CONFLICT, STUDENT_CONFLICT]",True
2,3,INSTRUCTOR_CONFLICT,INSTRUCTOR_CONFLICT,"[INSTRUCTOR_CONFLICT, STUDENT_CONFLICT, INELIG...",True
3,4,STUDENT_CONFLICT,STUDENT_CONFLICT,[STUDENT_CONFLICT],True
4,5,ROOM_TYPE_MISMATCH,ROOM_TYPE_MISMATCH,"[ROOM_TYPE_MISMATCH, EQUIPMENT_SHORTAGE]",True
5,6,CAPACITY_SHORTAGE,CAPACITY_SHORTAGE,[CAPACITY_SHORTAGE],True
6,7,EQUIPMENT_SHORTAGE,EQUIPMENT_SHORTAGE,[EQUIPMENT_SHORTAGE],True
7,8,INSTRUCTOR_UNAVAILABLE,INSTRUCTOR_UNAVAILABLE,"[INSTRUCTOR_UNAVAILABLE, ROOM_CONFLICT]",True
8,9,ROOM_CLOSED,ROOM_CLOSED,[ROOM_CLOSED],True
9,10,HOLIDAY,HOLIDAY,[HOLIDAY],True


All 12 fixture tests passed

TANSEEK MODEL FINAL CHECK ✅
Fixture tests: 12/12
Existing allocations: 0
Weekly sessions to solve: 12
Scheduled: 12
Unscheduled: 0
Term holidays loaded: 1
Room closures loaded: 1
JSON output: C:\Users\engme\OneDrive\Desktop\Tanseek_Menna_Model_v2_Starter\tanseek_solver_output.json
Post-solve hard conflicts: 0


In [106]:
# =================================================
# FINAL TANSEEK VALIDATION
# =================================================
# -------------------------
# Final Objective Metrics
# -------------------------

candidate_score_value = (
    int(selected_schedule["total_score"].sum())
    if not selected_schedule.empty
    else 0
)


selected_compact_pairs = int(
    sum(
        solver.Value(variable)
        for variable in compactness_vars
    )
)


compactness_bonus_value = (
    selected_compact_pairs
    * COMPACTNESS_POINTS
)


active_room_ids = [
    int(room_id)
    for room_id, variable in room_used.items()
    if solver.Value(variable) == 1
]


active_room_count = len(
    active_room_ids
)


room_utilization_penalty_value = (
    active_room_count
    * ROOM_ACTIVE_PENALTY
)


unscheduled_penalty_value = (
    len(unscheduled_sessions)
    * UNSCHEDULED_PENALTY
)


final_objective_value = (
    candidate_score_value
    + compactness_bonus_value
    - room_utilization_penalty_value
    - unscheduled_penalty_value
)


solver_objective_value = int(
    round(
        solver.ObjectiveValue()
    )
)


print(
    "Candidate score:",
    candidate_score_value
)

print(
    "Compact adjacent pairs:",
    selected_compact_pairs
)

print(
    "Compactness bonus:",
    compactness_bonus_value
)

print(
    "Active rooms:",
    active_room_count
)

print(
    "Room utilization penalty:",
    room_utilization_penalty_value
)

print(
    "Unscheduled penalty:",
    unscheduled_penalty_value
)

print(
    "Calculated objective:",
    final_objective_value
)

print(
    "Solver objective:",
    solver_objective_value
)


assert (
    final_objective_value
    == solver_objective_value
)

print(
    "Objective breakdown validated ✅"
)
print()
print("===================================")
print("TANSEEK FINAL VALIDATION")
print("===================================")


# -------------------------
# 1) Objective Breakdown
# -------------------------

assert (
    int(final_objective_value)
    == int(solver_objective_value)
), (
    f"Objective mismatch: "
    f"calculated={final_objective_value}, "
    f"solver={solver_objective_value}"
)

print("Objective breakdown validated ✅")


# -------------------------
# 2) Scheduled Payload
# -------------------------

assert isinstance(
    scheduled_payload,
    list
)

assert (
    len(scheduled_payload)
    == len(selected_schedule)
)

for item in scheduled_payload:

    assert "session_key" in item
    assert "score" in item
    assert "score_breakdown" in item
    assert "alternatives" in item

    breakdown = item[
        "score_breakdown"
    ]

    assert "staff_preference" in breakdown
    assert "capacity_fit" in breakdown
    assert "equipment_match" in breakdown
    assert "candidate_total" in breakdown


print("Scheduled payload ready ✅")


# -------------------------
# 3) Hard Conflict Check
# -------------------------

assert (
    len(post_solve_conflicts)
    == 0
), post_solve_conflicts

print("Post-solve hard conflicts: 0 ✅")


# -------------------------
# 4) Fixture Tests
# -------------------------

fixture_results = run_fixture_tests()

assert (
    fixture_results["pass"].all()
)

print(
    f"Fixture tests: "
    f"{fixture_results['pass'].sum()}/"
    f"{len(fixture_results)} ✅"
)


# -------------------------
# 5) Session Coverage
# -------------------------

scheduled_keys = set(
    selected_schedule[
        "session_key"
    ]
)

unscheduled_keys = set(
    unscheduled_sessions
)

required_keys = set(
    weekly_sessions[
        "session_key"
    ]
)


assert scheduled_keys.isdisjoint(
    unscheduled_keys
)

assert (
    scheduled_keys
    | unscheduled_keys
    == required_keys
)


print("Session coverage validated ✅")


# -------------------------
# 6) Output JSON
# -------------------------

assert OUTPUT_FILE.exists()

print("JSON output exists ✅")


# -------------------------
# FINAL RESULT
# -------------------------

print()
print("===================================")
print("TANSEEK MODEL FINAL CHECK ✅")
print("===================================")

print(
    "Existing allocations:",
    len(solver_existing_allocations)
)

print(
    "Weekly sessions to solve:",
    len(weekly_sessions)
)

print(
    "Scheduled:",
    len(selected_schedule)
)

print(
    "Unscheduled:",
    len(unscheduled_sessions)
)

print(
    "Candidate-level score:",
    candidate_score_value
)

print(
    "Compact adjacent pairs:",
    selected_compact_pairs
)

print(
    "Compactness bonus:",
    compactness_bonus_value
)

print(
    "Active rooms:",
    active_room_count
)

print(
    "Room utilization penalty:",
    room_utilization_penalty_value
)

print(
    "Final objective:",
    final_objective_value
)

print(
    "Term holidays loaded:",
    len(term_holidays)
)

print(
    "Room closures loaded:",
    len(room_closures)
)

print(
    "JSON output:",
    OUTPUT_FILE.resolve()
)

print("===================================")

Candidate score: 300
Compact adjacent pairs: 4
Compactness bonus: 60
Active rooms: 4
Room utilization penalty: 40
Unscheduled penalty: 0
Calculated objective: 320
Solver objective: 320
Objective breakdown validated ✅

TANSEEK FINAL VALIDATION
Objective breakdown validated ✅
Scheduled payload ready ✅
Post-solve hard conflicts: 0 ✅


,test_case_id,expected,actual,all_reasons,pass
0,1,NONE,FEASIBLE,[],True
1,2,ROOM_CONFLICT,ROOM_CONFLICT,"[ROOM_CONFLICT, STUDENT_CONFLICT]",True
2,3,INSTRUCTOR_CONFLICT,INSTRUCTOR_CONFLICT,"[INSTRUCTOR_CONFLICT, STUDENT_CONFLICT, INELIG...",True
3,4,STUDENT_CONFLICT,STUDENT_CONFLICT,[STUDENT_CONFLICT],True
4,5,ROOM_TYPE_MISMATCH,ROOM_TYPE_MISMATCH,"[ROOM_TYPE_MISMATCH, EQUIPMENT_SHORTAGE]",True
5,6,CAPACITY_SHORTAGE,CAPACITY_SHORTAGE,[CAPACITY_SHORTAGE],True
6,7,EQUIPMENT_SHORTAGE,EQUIPMENT_SHORTAGE,[EQUIPMENT_SHORTAGE],True
7,8,INSTRUCTOR_UNAVAILABLE,INSTRUCTOR_UNAVAILABLE,"[INSTRUCTOR_UNAVAILABLE, ROOM_CONFLICT]",True
8,9,ROOM_CLOSED,ROOM_CLOSED,[ROOM_CLOSED],True
9,10,HOLIDAY,HOLIDAY,[HOLIDAY],True


All 12 fixture tests passed
Fixture tests: 12/12 ✅
Session coverage validated ✅
JSON output exists ✅

TANSEEK MODEL FINAL CHECK ✅
Existing allocations: 0
Weekly sessions to solve: 12
Scheduled: 12
Unscheduled: 0
Candidate-level score: 300
Compact adjacent pairs: 4
Compactness bonus: 60
Active rooms: 4
Room utilization penalty: 40
Final objective: 320
Term holidays loaded: 1
Room closures loaded: 1
JSON output: C:\Users\engme\OneDrive\Desktop\Tanseek_Menna_Model_v2_Starter\tanseek_solver_output.json
